## 4. SMAP soil moisture animation — 9 km, 3 km and 1 km

Animate the monthly-mean soil moisture across the 2025 growing season (April-October) at the three resolutions, reading the GeoTIFF files from the Teams folder. The maps are rasters, drawn as pixels. Coverage is most complete at 9 km and sparsest at 1 km; spatial detail increases from 9 km to 1 km.

## 0. Setup


In [ ]:
!pip install -q geemap rasterio

In [ ]:
import ee, geemap

PROJECT_ID = "your-project-id"     # your Earth Engine project id

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

Map = geemap.Map(center=[41.7, -91.5], zoom=7)
Map

In [ ]:
# Areas of interest (edit to study your own region)
AG_AOI    = ee.Geometry.Rectangle([-93.9, 41.9, -93.2, 42.5])   # central Iowa
FLOOD_AOI = ee.Geometry.Rectangle([-96.3, 40.4, -95.4, 41.4])   # Missouri River, NE/IA
CORN_BELT = (-99, 37, -82, 46)                                  # lon W, lat S, lon E, lat N

## 1. Sentinel-1 backscatter and a false-colour composite

Sentinel-1 records two polarizations in decibels (dB): VV (sensitive to roughness and open water) and VH (cross-pol, sensitive to vegetation). Viewing VV, VH and their difference as the red, green and blue channels gives a false-colour image where water, fields and built-up areas take on distinct colours.


In [ ]:
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(AG_AOI).filterDate('2025-06-01', '2025-06-30')
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')))

img = s1.median().clip(AG_AOI)
vv, vh = img.select('VV'), img.select('VH')
composite = ____   # TODO: stack VV, VH and VV-VH as the R, G, B channels

Map = geemap.Map(center=[42.2, -93.55], zoom=10)
Map.addLayer(vv, {'min': -25, 'max': 0},  'VV (dB)')
Map.addLayer(vh, {'min': -30, 'max': -5}, 'VH (dB)')
Map.addLayer(composite, {'bands': ['R', 'G', 'B'],
             'min': [-25, -30, 0], 'max': [0, -5, 15]}, 'False-colour (VV, VH, VV-VH)')
Map

## 2. Radar Vegetation Index (RVI)

RVI estimates vegetation density from the two channels. It is defined on linear power, so convert from dB first: sigma0 = 10^(dB/10). Low RVI is bare/smooth ground; high RVI is dense canopy.


In [ ]:
lin = ee.Image(10).pow(img.divide(10))     # dB -> linear power
vv, vh = lin.select('VV'), lin.select('VH')
rvi = ____   # TODO: 4 * VH / (VV + VH)

Map = geemap.Map(center=[42.2, -93.55], zoom=10)
Map.addLayer(rvi, {'min': 0, 'max': 1,
             'palette': ['#8c510a','#d8b365','#f6e8c3','#c7eae5','#5ab4ac','#01665e']}, 'RVI')
Map

## 3. Flood mapping — March 2019 Missouri River

Open water reflects radar away from the sensor, so flooded ground looks dark. Compare a before and after VH composite and flag pixels that became much darker. Sentinel-1 covers this area on 22 Feb (before) and 18 Mar (peak flood).


In [ ]:
def s1_vh(aoi, d0, d1):
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(aoi).filterDate(d0, d1)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .select('VH').median().clip(aoi))

pre   = s1_vh(FLOOD_AOI, '2019-02-15', '2019-02-28')
post  = s1_vh(FLOOD_AOI, '2019-03-15', '2019-03-22')
flood = ____   # TODO: dark now (post < -18) AND a big drop (post - pre < -3)

Map = geemap.Map(center=[40.95, -95.85], zoom=10)
Map.addLayer(pre,  {'min': -30, 'max': -5}, 'Before VH (22 Feb 2019)')
Map.addLayer(post, {'min': -30, 'max': -5}, 'After VH (18 Mar 2019)')
Map.addLayer(flood.selfMask(), {'palette': ['#1f6feb']}, 'Flooded')
Map

## 4. SMAP soil moisture at 9 km, 3 km and 1 km

Monthly-mean soil moisture (June 2025) over the Corn Belt at the three resolutions, read from the GeoTIFF files in the Teams folder. The maps are rasters, drawn as pixels. Coverage is most complete at 9 km and sparsest at 1 km (which needs a Sentinel-1 overpass); spatial detail increases from 9 km to 1 km.


In [ ]:
import os, numpy as np, rasterio
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from IPython.display import HTML

# Works whether the data sits in ./Teams_folder or in the same folder as this notebook
DATA_DIR = 'Teams_folder' if os.path.isdir('Teams_folder/SMAP_9km') else '.'
months = ['04', '05', '06', '07', '08', '09', '10']
names  = ['Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct']
products = [('SMAP 9 km',            'SMAP_9km/GeoTIFF/SMAP_9km_2025{m}.tif'),
            ('SMAP/Sentinel-1 3 km', 'SMAP_Sentinel1_3km/GeoTIFF/SMAP_S1_3km_2025{m}.tif'),
            ('SMAP/Sentinel-1 1 km', 'SMAP_Sentinel1_1km/GeoTIFF/SMAP_S1_1km_2025{m}.tif')]

def read(path):
    with rasterio.open(path) as src:
        a = src.read(1).astype('float32'); b = src.bounds; nd = src.nodata
    return np.where(a == nd, np.nan, a), [b.left, b.right, b.bottom, b.top]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ims = []
for ax, (label, tmpl) in zip(axes, products):
    a, ext = read(f"{DATA_DIR}/{tmpl.format(m=months[0])}")
    ims.append(ax.imshow(a, cmap='YlGnBu', vmin=0, vmax=0.6, extent=ext, interpolation='nearest'))
    ax.set_title(label); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

cax = fig.add_axes([0.32, -0.02, 0.36, 0.028])
fig.colorbar(ScalarMappable(Normalize(0, 0.6), plt.get_cmap('YlGnBu')),
             cax=cax, orientation='horizontal').set_label('Volumetric soil moisture (m3/m3)   dry to wet')
title = fig.suptitle('', y=1.0, fontsize=15)

def update(i):
    for im, (label, tmpl) in zip(ims, products):
        a, _ = read(f"{DATA_DIR}/{tmpl.format(m=months[i])}")
        im.set_data(a)
    title.set_text(f'SMAP soil moisture - {names[i]} 2025, U.S. Corn Belt')
    return ims

ani = animation.FuncAnimation(fig, update, frames=len(months), interval=900, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())     # inline animation player

## 5. Try it on other events — customize the AOI and dates

The flood method is just change detection, so it works for any event that changes radar backscatter. Reuse the same logic but change the area of interest and the two date windows. Sentinel-1 covers the whole globe; print the collection size to confirm your dates have scenes.

| Event | AOI [W, S, E, N] | before | after |
|---|---|---|---|
| 2022 Pakistan (Indus) floods | [68.0, 26.5, 69.5, 28.0] | 2022-07-01 to 07-31 | 2022-08-25 to 09-12 |
| 2022 Lismore, Australia floods | [153.0, -28.9, 153.4, -28.6] | 2022-02-10 to 02-25 | 2022-03-01 to 03-12 |
| 2021 La Palma volcano (lava) | [-17.95, 28.55, -17.80, 28.72] | 2021-09-01 to 09-15 | 2021-10-20 to 11-05 |

For floods, new water is dark (backscatter drops). For fresh lava or newly bare ground, backscatter usually rises, so flip the test to a positive change.


In [ ]:
def change_map(aoi_bbox, before_d0, before_d1, after_d0, after_d1,
               dark_db=-18, drop_db=-3):
    aoi = ee.Geometry.Rectangle(aoi_bbox)
    n = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi)
         .filterDate(before_d0, before_d1).size().getInfo())
    print('before scenes found:', n, '(0 means no coverage - change the dates)')
    before = s1_vh(aoi, before_d0, before_d1)
    after  = s1_vh(aoi, after_d0,  after_d1)
    change = after.lt(dark_db).And(after.subtract(before).lt(drop_db))
    return aoi, before, after, change

# 2022 Pakistan (Indus) floods. Copy this cell and swap in another row from the table above.
aoi, before, after, change = change_map(
    [68.0, 26.5, 69.5, 28.0], '2022-07-01', '2022-07-31', '2022-08-25', '2022-09-12')

Map = geemap.Map()
Map.addLayer(before, {'min': -30, 'max': -5}, 'Before VH')
Map.addLayer(after,  {'min': -30, 'max': -5}, 'After VH')
Map.addLayer(change.selfMask(), {'palette': ['#1f6feb']}, 'Change (new water)')
Map.centerObject(aoi, 9); Map

The original HDF5 products and their bands, quality flags and metadata are described in README_product_notes.txt in the Teams folder. The full download-and-processing workflow is in Workshop_final_version.ipynb.
